# **Population-Scale Homeostatic Simulations**

# **Fyon Model — Heterogeneous Population (200 neurons)**

This notebook investigates homeostatic compensatory mechanisms 
in a heterogeneous population of 200 dopaminergic neurons 
following targeted pharmacological blockades.

**Simulation protocol :**
- Baseline phase : 0 → 30 s
- Blockade applied instantaneously at t = 30 s
- Compensation observed until t = 300 s

**Blockades simulated :**
1. L-type calcium channels (Cav1.3) — isradipine strategy
2. HCN channels — validation (should preserve pacemaking)
3. A-type potassium channels — validation (should preserve pacemaking)
4. gPace conductance — role of IXG current
5. gPace + synaptic noise — network effect

# **Useful packages and functions**

In [ ]:
using DifferentialEquations, Plots, Plots.PlotMeasures, LaTeXStrings, Random, Dierckx, DelimitedFiles
using OrdinaryDiffEq
using ProgressMeter
using Interpolations
using NmodController
using Statistics
include("DA_kinetics7.jl") 
include("Essaie7.jl")
include("STG_utils7.jl")

# **Global variables**

In [ ]:
Tfinal = 300000
tspan  = (0.0, Tfinal)
tt = 0. : 0.01 : Tfinal
tt_rand = 0. : 1 : Tfinal

gr(guidefontsize=18, legendfontsize=12, margin=5Plots.mm, grid=false)
myApple = RGBA(187/255, 206/255, 131/255, 1)
mySalmon = RGBA(243/255, 124/255, 130/255)
myYellow = RGBA(228/255, 205/255, 121/255, 1)
myBlue = RGBA(131/255, 174/255, 218/255, 1)
myDarkBlue = RGBA(114/255, 119/255, 217/255, 1)
myOrange = RGBA(241/255, 175/255, 113/255, 1)
myPink = RGBA(243/255, 124/255, 130/255, 1)
myPurple = RGBA(169/255, 90/255, 179/255, 1)
myGreen = RGBA(132/255, 195/255, 168/255, 1)
myRed = RGBA(158/255, 3/255, 8/255, 1)
myGray = RGBA(150/255, 150/255, 150/255, 1)
myLightBlue = RGBA(127/255, 154/255, 209/255, 1)
default(fmt = :png)

moving_average(vs, n, padding) = [sum(vs[i:(i+n-1)])/n for i in 1:padding:(length(vs)-(n-1))];

In [ ]:
# Definition of reversal potential values (in mV), [Mg] and membrane capacitance
const VNa     = 60. # Sodium reversal potential
const VK      = -90. # Potassium reversal potential
const VCa     = 50. # Calcium reversal potential
const VH      = -29. # H reversal potential
const VLNS    = -65. # Leak reversal potential
const EPacemaker = 4.2732015978991615 # Reversal potential of pacemaking channels

const C       = 1. # Membrane capacitance
const fCa     = 0.018 # Fraction of unbuffered free calcium
const ICapmax = 11 # Maximum calcium pump current
const F       = 96520 # Faraday constant in ms*µA/mmol (and taking cm³=mL)
const d       = 15 # Soma diameter in cm
const L       = 25 # Soma length

const Vmin = -100 
const Vmax = 50
const Vrange = range(Vmin, stop=Vmax, step=0.0154640);

# **Population Setup and Calibration**

The 200-neuron population was pre-generated by Fyon et al.via random sampling in conductance space, retaining only neurons 
with pacemaking activity (1–5 Hz).

In [ ]:
#Chargement des conductances pré-calculées
g_all_init = readdlm("g_all_pacemaking.dat", '\t', Float64)

#ncells = size(g_all_init, 1) 
println("Chargement réussi :  neurones prêts.")

# **Calibration of gCaT (T-type conductance)**

For each neuron, gCaT is calibrated so that T-type channels 
contribute exactly 10% of the total calcium influx per 
pacemaking cycle. This ensures a meaningful alternative 
calcium source when L-type channels are blocked.

In [ ]:
#Definition de la fonction qui va permettre de définir la valeur de conductance de gTtype 
# L'idée est de s'assurer que quand on bloque gCaL, le point de départ 
#du T-type est déjà suffisamment significatif pour que la compensation homéostatique reste dans une gamme raisonnable

function calibrate_gTtype(sol, p_init; target_ratio=0.1)
    # 1. Détection des spikes pour isoler un cycle
    gCaL = p_init[3]
    gCaT_fallback = 0.1 * gCaL 
    threshold = gCaT_fallback
    v_trace = sol[1, :]
    times = sol.t
    spike_indices = findall(i -> v_trace[i-1] < 0 && v_trace[i] >= 0, 2:length(v_trace))
    
    if length(spike_indices) < 2
        return 0.2 
    end
    
    # On prend le dernier cycle complet pour la stabilité
    idx1, idx2 = spike_indices[end-1], spike_indices[end]
    tt = range(times[idx1], times[idx2], length=1000)
    res = sol(tt)
    V = res[1, :]
    dt = tt[2] - tt[1]
    
    gCaL   = p_init[3]
    gLCa   = p_init[10]
    gNtype = p_init[12]
    
    # Calculer les intégrales des courants 
    # Q = g * ∫ (gating * (V-VCa)) dt
    # On calcule A = ∫ (gating * (V-VCa)) dt

    # 1. L-type 
    l = res[5, :]
    A_CaL = sum(l .* (V .- VCa)) * dt
    
    # 2. N-type
    mN = res[14, :]
    hN = res[15, :]
    A_CaN = sum((mN .^ 2) .* hN .* (V .- VCa)) * dt
    
    # 3. Fuite Ca
    A_LCa = sum(V .- VCa) * dt
    
    # 4. T-type 
    mT = res[16, :]
    hT = res[17, :]
    A_CaT = sum(mT .* hT .* (V .- VCa)) * dt
    
    # Équation de calibration :
    # gT * A_CaT / (gCaL * A_CaL + gLCa * A_LCa + gT * A_CaT + gN * A_CaN) = target_ratio
    num = target_ratio * (gCaL * A_CaL + gNtype * A_CaN + gLCa * A_LCa)
    
    den = A_CaT * (1 - target_ratio)
    
    gT_calibrated = abs(num / den)
    print(gT_calibrated)
    
    return gT_calibrated
end
    

In [ ]:
ncells = 200
gTtype_final=zeros(ncells)

@showprogress "Computing Population..." for i = 1 : ncells
    # 1. Initialisation classique des paramètres
    gNa = g_all_init[i,1]
    gCaL = g_all_init[i,2]
    gKd = g_all_init[i,3]
    gKA = g_all_init[i,4]
    gKERG = g_all_init[i,5]
    gKSK = g_all_init[i,6]
    gH = g_all_init[i,7]
    gLNS = g_all_init[i,8]
    gLCa = g_all_init[i,9]
    gPacemaker = g_all_init[i,10]
    gNtype = 0.2*(0.9 + 0.2 * rand()) 
    gTtype = 0.025 
    
    Iapp(t) = 0
    p_calib = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker, gNtype, gTtype)
    V0 = -50.
    Ca0 = 1e-4
    x0 = [V0, m_inf_true(V0), h_inf(V0), hs_inf(V0), l_inf_true(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
         0., 0., mH_inf(V0), Ca0, mN_inf(V0), hN_inf(V0), mT_inf(V0), hT_inf(V0)]
    tspan_calib = (0.0, 1000.0)
    prob_calib = ODEProblem(DA_ODE_true_instant, x0, tspan_calib, p_calib) 
    sol_calib = solve(prob_calib,maxiters=1e6)
    gTtype_final[i] = calibrate_gTtype(sol_calib, p_calib; target_ratio=0.1)
end



# **Calibration of individual calcium targets (Ca_tgt)**

Each neuron has a unique resting calcium concentration depending on its initial conductance profile. Ca_tgt is 
defined as the mean [Ca²⁺] during a free-running simulation (no homeostatic controller, no blockade).

In [ ]:
ncells= 200
Tfinal = 100000
tt = 0.0 : 0.2 : Tfinal
tspan  = (0.0, Tfinal)

Ca_tgt_moyen =zeros(ncells)

@showprogress "Computing Population..." for i = 1 : ncells
    gNa = g_all_init[i,1]
    gCaL = g_all_init[i,2]
    gKd = g_all_init[i,3]
    gKA = g_all_init[i,4]
    gKERG = g_all_init[i,5]
    gKSK = g_all_init[i,6]
    gH = g_all_init[i,7]
    gLNS = g_all_init[i,8]
    gLCa = g_all_init[i,9]
    gPacemaker = g_all_init[i,10]
    gNtype = 0.2*(0.9 + 0.2 * rand()) 
    gTtype = gTtype_final[i]
    
    Iapp(t) = 0 
    p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker, gNtype, gTtype)
    V0 = -50.
    Ca0 = 1e-4
    x0 = [V0, m_inf_true(V0), h_inf(V0), hs_inf(V0), l_inf_true(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
         0., 0., mH_inf(V0), Ca0, mN_inf(V0), hN_inf(V0), mT_inf(V0), hT_inf(V0)]
    prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) 
    sol = solve(prob,maxiters=1e6);
    x_res  = sol(tt)
    Ca_mean = mean(x_res[13, :])

    Ca_tgt_moyen[i] = Ca_mean
    
end


In [ ]:
# 1. Calcule le log10 des cibles (assure-toi qu'il n'y a pas de <= 0)
log_data = log10.(Ca_tgt_moyen[Ca_tgt_moyen .> 0])

# 2. Plot en mode linéaire (puisque les données sont déjà en log)
p_ca_tgt = violin(log_data, 
             label="", 
             color=myGray, 
             grid=false,
             xticks=([1,], ["Population",]),
             yticks=([-5, -4, -3, -2, -1, 0], 
                    [L"10^{-5}", L"10^{-4}", L"10^{-3}", L"10^{-2}", L"10^{-1}", L"1"]), size=(400, 500))

ylabel!("Calcium Target (mM)")

# **Blockade Simulations**

> **Protocol** : Each simulation runs for 300 s. The target 
> conductance is set to 1e-18 at t = 30 s (instantaneous 
> pharmacological blockade). The homeostatic controller then 
> adjusts remaining conductances to recover Ca_tgt.

# **L-type Calcium Channel Blockade (Cav1.3)**

In [ ]:
# 3. Lancement de la population

Tfinal = 300000
tt = 0.0 : 0.2 : Tfinal
tspan  = (0.0, Tfinal)

window_size_s = 5.0       
padding_s = 0.2           
window_size_  = Int64(round(window_size_s * 1000 / 0.2)) 
padding_      = Int64(round(padding_s * 1000 / 0.2))
tt_moving_average_plot = range(window_size_s/2, Tfinal/1000 - window_size_s/2, length=length(moving_average(zeros(length(tt)), window_size_, padding_)))

tau_g = 0.1
t_Na = 0.6

ncells= 200
Tfinal = tspan[2]           

sample_indices = 2:5000:length(tt)
data_length = length(sample_indices)

gNa_matrix = zeros(ncells, data_length)
gCaL_matrix = zero(gNa_matrix)
gKd_matrix = zero(gNa_matrix)
gKA_matrix = zero(gNa_matrix)
gKERG_matrix = zero(gNa_matrix)
gKSK_matrix = zero(gNa_matrix)
gH_matrix = zero(gNa_matrix)
gPace_matrix = zero(gNa_matrix)
gLNS_matrix = zero(gNa_matrix)
gLCa_matrix = zero(gNa_matrix)
gNtype_matrix = zero(gNa_matrix)
gTtype_matrix = zero(gNa_matrix)

freqs_pre = zeros(ncells)
freqs_acute = zeros(ncells)
freqs_comp = zeros(ncells)
ca_pre_population = zeros(ncells)
ca_acute_population = zeros(ncells)
ca_comp_population = zeros(ncells)

win_pre   = (5000, 25000)    # Avant blocage (30s)
win_acute = (35000, 55000)   # Juste après le blocage
win_comp  = (250000, 295000) # À la fin 

Ca_ma_matrix = zeros(ncells, length(tt_moving_average_plot))

@showprogress "Computing Population..." for i = 1 : ncells
    gNa = g_all_init[i,1]
    gCaL = g_all_init[i,2]
    gKd = g_all_init[i,3]
    gKA = g_all_init[i,4]
    gKERG = g_all_init[i,5]
    gKSK = g_all_init[i,6]
    gH = g_all_init[i,7]
    gLNS = g_all_init[i,8]
    gLCa = g_all_init[i,9]
    gPace = g_all_init[i,10]
    gNtype = 0.2*(0.9 + 0.2 * rand())
    gTtype = gTtype_final[i]
    
    Ca_tgt = Ca_tgt_moyen[i]
    
    tCaL   = t_Na * gNa / max(1e-6,gCaL)
    tKd    = t_Na * gNa / max(1e-6,gKd)
    tKA    = t_Na * gNa / max(1e-6,gKA)
    tKERG  = t_Na * gNa / max(1e-6,gKERG) 
    tKSK   = t_Na * gNa / max(1e-6,gKSK)
    tH     = t_Na * gNa / max(1e-6,gH)
    tPace  = t_Na * gNa / max(1e-6,gPace)
    tLNS   = t_Na * gNa / max(1e-6,gLNS)
    tLCa   = t_Na * gNa / max(1e-6,gLCa)
    tNCa   = t_Na * gNa / max(1e-6,gNtype)
    tTCA   = t_Na * gNa / max(1e-6,gTtype)
    
    Iapp_val(t) = 0.0
    
    g_main_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH,gPace]
    m_main_init = copy(g_main_init)

    g_leak_init = [gLNS,  gLCa]
    m_leak_init = copy(g_leak_init)

    g_calcique = [gNtype, gTtype]
    m_calcique = copy(g_calcique)
    
    g_max_main = g_main_init .* 10.0
    g_max_leak = g_leak_init .* 10.0
    g_max_cal  = g_calcique  .* 10.0

    g_max_all = (g_max_main, g_max_leak, g_max_cal)

    p_homeo = (Iapp_val, Ca_tgt, tau_g, t_Na, tCaL, tKd, tKA, tKERG, tKSK, 
           tH, tPace, tLNS, tLCa, tNCa, tTCA, g_max_all)

    V0 = -50.0
    Ca0 = Ca_tgt_moyen[i]

    u_biophys = [
        V0, m_inf_true(V0), h_inf(V0), hs_inf(V0), l_inf_true(V0), n_inf(V0), 
        p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
    ]

    uphys = [mN_inf(V0), hN_inf(V0), mT_inf(V0), hT_inf(V0)]

    u0_homeo = vcat(u_biophys, g_main_init, m_main_init, g_leak_init, m_leak_init, g_calcique, m_calcique, uphys)

    prob = ODEProblem(DA_homeo_ODE_Ntype_Ttype_FixedLeak_Ltype, u0_homeo, tspan, p_homeo)
    
    sol_rigorous = solve(prob, Rodas5P(), reltol=1e-4, maxiters=1e7)
    
    if sol_rigorous.t[end] < Tfinal
        @warn "Crash persistant à t = $(sol_rigorous.t[end])"
    end
    
    t_echantillon = tt[sample_indices]
    temp_sol = sol_rigorous(t_echantillon)
    
    gNa_matrix[i, :]   = temp_sol[14, :]
    gCaL_regulee = temp_sol[15, :]
    gCaL_matrix[i, :] = [ (t_echantillon[j] < 30000.0) ? gCaL_regulee[j] : 1e-18 for j in 1:length(t_echantillon) ]
    gKd_matrix[i, :]   = temp_sol[16, :]
    gKA_matrix[i, :]   = temp_sol[17, :]
    gKERG_matrix[i, :] = temp_sol[18, :]
    gKSK_matrix[i, :]  = temp_sol[19, :]
    gH_matrix[i, :]    = temp_sol[20, :]
    gPace_matrix[i, :] = temp_sol[21, :]
    gLNS_matrix[i, :]  = temp_sol[30, :]
    gLCa_matrix[i, :]  = temp_sol[31, :]
    
    gNtype_matrix[i, :] = temp_sol[34, :]
    gTtype_matrix[i, :] = temp_sol[35, :]
    Ca_ma_matrix[i, :] = moving_average(sol_rigorous(tt)[13, :], window_size_, padding_)
    
    # Extraction et calculs
    t_pre = win_pre[1]:0.1:win_pre[2]  
    sol_pre = sol_rigorous(t_pre)      
    f_pre = extract_frequency(sol_pre[1, :], t_pre)
    freqs_pre[i] = isnan(f_pre) ? 0.0 : f_pre 
    ca_pre_population[i] = mean(sol_pre[13, :]) 

    t_acute = win_acute[1]:0.1:win_acute[2]
    sol_acute = sol_rigorous(t_acute)
    f_acute = extract_frequency(sol_acute[1, :], t_acute)
    freqs_acute[i] = isnan(f_acute) ? 0.0 : f_acute
    ca_acute_population[i] = mean(sol_acute[13, :]) 

    t_comp = win_comp[1]:0.1:win_comp[2]
    sol_comp = sol_rigorous(t_comp)
    f_comp = extract_frequency(sol_comp[1, :], t_comp)
    freqs_comp[i] = isnan(f_comp) ? 0.0 : f_comp
    ca_comp_population[i] = mean(sol_comp[13, :])
end
    
   


In [ ]:
# Filtre les 3 neurones crashés
valid = (ca_pre_population .> 0) .& 
        (ca_acute_population .> 0) .& 
        (ca_comp_population .> 0)

log_ca_pre   = log10.(ca_pre_population[valid])
log_ca_acute = log10.(ca_acute_population[valid])
log_ca_comp  = log10.(ca_comp_population[valid])

y_ticks_val = [-6, -5, -4, -3]
y_ticks_lab = [L"10^{-6}", L"10^{-5}", L"10^{-4}", L"10^{-3}"]

# bandwidth=0.3 → c'est LE paramètre qui adoucit les bords
vc1 = violin([log_ca_pre], label="", color=myApple, grid=false, 
             title="Pre-block Calcium", npoints=800, bandwidth=0.3,trim= false,
             xticks=([1,], [L"t_\mathrm{pre}",]), 
             yticks=(y_ticks_val, y_ticks_lab), ylims=(-6.5, -2.5))

vc2 = violin([log_ca_acute], label="", color=:red, grid=false, 
             title="Acute Calcium", npoints=800, bandwidth=0.3,trim= false,
             xticks=([1,], [L"t_\mathrm{acute}",]), 
             yticks=(y_ticks_val, y_ticks_lab), ylims=(-6.5, -2.5))

vc3 = violin([log_ca_comp], label="", color=myTeal, grid=false, 
             title="Compensated Calcium", npoints=800, bandwidth=0.3,trim= false,
             xticks=([1,], [L"t_\mathrm{comp}",]), 
             yticks=(y_ticks_val, y_ticks_lab),ylims=(-7.2, -2.0))

p_calcium = plot(vc1, vc2, vc3, layout=(1,3), size=(1100, 400), 
                 ylabel="Intracellular Calcium (mM)",
                 link=:y)
display(p_calcium)

In [ ]:
# --- 1. FILTRAGE DE FREQS_PRE ---
# On ne garde que les neurones qui spikent à moins de 5 Hz en condition initiale
freqs_pre_filtered = freqs_pre[freqs_pre .<= 5.0]

# --- 2. CALCUL DES POURCENTAGES ---
n_total = length(freqs_pre)
pc_pre   = count(f -> 0.5 < f <= 5.0, freqs_pre) / n_total * 100
pc_acute = count(is_valid_pm, freqs_acute) / n_total * 100
pc_comp  = count(is_valid_pm, freqs_comp) / n_total * 100

# --- 3. LES VIOLIN PLOTS ---
v1 = violin([freqs_pre_filtered], label="", color=myApple, grid=false,
            title="Pre-block  : $pc_pre%",
            xticks=([1,], [L"t_\mathrm{pre}",]), ylims=(0, 15))

v2 = violin([freqs_acute], label="", color=:red, grid=false,
            title="Acute (50sec) : $pc_acute%",
            xticks=([1,], [L"t_\mathrm{acute}",]), ylims=(0, 15))

v3 = violin([freqs_comp], label="", color=myTeal, grid=false,
            title="Compensated (300sec): $pc_comp%",
            xticks=([1,], [L"t_\mathrm{comp}",]), ylims=(0, 15))

plot(v1, v2, v3, layout=(1,3), size=(1100, 400), ylabel="Frequency (Hz)")

# 1. Neurones qui ont migré de la haute fréquence (10-15Hz) vers une fréquence centrale (~5Hz)
id_shift = findall(i -> (10.0 <= freqs_acute[i] <= 15.0) && (4.0 <= freqs_comp[i] <= 6.0), 1:n_simulated)
println("IDs des neurones ayant shifté de 10-15Hz vers ~5Hz : ", id_shift)

# On cherche les neurones dont la fréquence a augmenté significativement entre l'aigu et la fin
id_ameliore = findall(i -> freqs_comp[i] > freqs_acute[i] + 0.5, 1:n_simulated)
println("IDs des neurones dont la fréquence a été boostée par l'homéostasie : ", id_ameliore)

In [ ]:
#plot calcium neurone individuel 
# --- CHOIX DU NEURONE ---
idx = 7 

# --- PLOT CALCIUM ---
p_ca = plot(size=(800, 400), title="Calcium Convergence - Neuron $idx")

# Courbe du calcium 
plot!(tt_moving_average_plot, Ca_ma_matrix[idx, :], 
      linewidth=2, color=:black, label="[Ca] moyen")
# Ligne cible
hline!([Ca_tgt_moyen[idx]], color=:firebrick1, linestyle=:dash, 
       linewidth=2, label="Calcium Target")

ylabel!(L"[Ca]\,\,(\mathrm{mM})")
xlabel!(L"t\,\mathrm{(s)}")
yaxis!(:log) 

display(p_ca)

# **HCN Channel Blockade — Model Validation**

In [ ]:
# 3. Lancement de la population
Tfinal = 300000
tt = 0.0 : 0.2 : Tfinal
tspan  = (0.0, Tfinal)

window_size_s = 5.0       
padding_s = 0.2           
window_size_  = Int64(round(window_size_s * 1000 / 0.2)) 
padding_      = Int64(round(padding_s * 1000 / 0.2))
tt_moving_average_plot = range(window_size_s/2, Tfinal/1000 - window_size_s/2, length=length(moving_average(zeros(length(tt)), window_size_, padding_)))

tau_g = 0.1
t_Na = 0.6

ncells= 200
Tfinal = tspan[2]           

sample_indices = 2:5000:length(tt)
data_length = length(sample_indices)

gNa_matrix = zeros(ncells, data_length)
gCaL_matrix = zero(gNa_matrix)
gKd_matrix = zero(gNa_matrix)
gKA_matrix = zero(gNa_matrix)
gKERG_matrix = zero(gNa_matrix)
gKSK_matrix = zero(gNa_matrix)
gH_matrix = zero(gNa_matrix)
gPace_matrix = zero(gNa_matrix)
gLNS_matrix = zero(gNa_matrix)
gLCa_matrix = zero(gNa_matrix)
gNtype_matrix = zero(gNa_matrix)
gTtype_matrix = zero(gNa_matrix)

freqs_pre = zeros(ncells)
freqs_acute = zeros(ncells)
freqs_comp = zeros(ncells)

win_pre   = (5000, 25000)    
win_acute = (35000, 55000)   
win_comp  = (250000, 295000) 

Ca_ma_matrix = zeros(ncells, length(tt_moving_average_plot))

@showprogress "Computing Population..." for i = 1 : ncells
    gNa = g_all_init[i,1]
    gCaL = g_all_init[i,2]
    gKd = g_all_init[i,3]
    gKA = g_all_init[i,4]
    gKERG = g_all_init[i,5]
    gKSK = g_all_init[i,6]
    gH = g_all_init[i,7]
    gLNS = g_all_init[i,8]
    gLCa = g_all_init[i,9]
    gPace = g_all_init[i,10]
    gNtype = 0.2 
    gTtype = gTtype_final[i]
    
    Ca_tgt = Ca_tgt_moyen[i]
    
    tCaL   = t_Na * gNa / max(1e-6,gCaL)
    tKd    = t_Na * gNa / max(1e-6,gKd)
    tKA    = t_Na * gNa / max(1e-6,gKA)
    tKERG  = t_Na * gNa / max(1e-6,gKERG) 
    tKSK   = t_Na * gNa / max(1e-6,gKSK)
    tH     = t_Na * gNa / max(1e-6,gH)
    tPace  = t_Na * gNa / max(1e-6,gPace)
    tLNS   = t_Na * gNa / max(1e-6,gLNS)
    tLCa   = t_Na * gNa / max(1e-6,gLCa)
    tNCa   = t_Na * gNa / max(1e-6,gNtype)
    tTCA   = t_Na * gNa / max(1e-6,gTtype)
    
    Iapp_val(t) = 0.0
    
    g_main_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH,gPace]
    m_main_init = copy(g_main_init)

    g_leak_init = [gLNS,  gLCa]
    m_leak_init = copy(g_leak_init)

    g_calcique = [gNtype, gTtype]
    m_calcique = copy(g_calcique)
    
    g_max_main = g_main_init .* 10.0
    g_max_leak = g_leak_init .* 10.0
    g_max_cal  = g_calcique  .* 10.0

    g_max_all = (g_max_main, g_max_leak, g_max_cal)

    p_homeo = (Iapp_val, Ca_tgt, tau_g, t_Na, tCaL, tKd, tKA, tKERG, tKSK, 
           tH, tPace, tLNS, tLCa, tNCa, tTCA, g_max_all)

    V0 = -50.0
    Ca0 = Ca_tgt_moyen[i]
    
    u_biophys = [
        V0, m_inf_true(V0), h_inf(V0), hs_inf(V0), l_inf_true(V0), n_inf(V0), 
        p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
    ]

    uphys = [mN_inf(V0), hN_inf(V0), mT_inf(V0), hT_inf(V0)]

    u0_homeo = vcat(u_biophys, g_main_init, m_main_init, g_leak_init, m_leak_init, g_calcique, m_calcique, uphys)

    prob = ODEProblem(DA_homeo_ODE_Hblock, u0_homeo, tspan, p_homeo)
    
    sol_rigorous = solve(prob, Rodas5P(), reltol=1e-4, maxiters=1e7)
    
    if sol_rigorous.t[end] < Tfinal
        @warn "Crash persistant à t = $(sol_rigorous.t[end])"
    end
    
    t_echantillon = tt[sample_indices]
    temp_sol = sol_rigorous(t_echantillon)
    
    gNa_matrix[i, :]   = temp_sol[14, :]
    gCaL_matrix[i, :]  = temp_sol[15, :]
    gKd_matrix[i, :]   = temp_sol[16, :]
    gKA_matrix[i, :]   = temp_sol[17, :]
    gKERG_matrix[i, :] = temp_sol[18, :]
    gKSK_matrix[i, :]  = temp_sol[19, :]
    gH_regulee = temp_sol[20, :]
    gH_matrix[i, :] = [ (t_echantillon[j] < 30000.0) ? gH_regulee[j] : 1e-18 for j in 1:length(t_echantillon) ]
    gPace_matrix[i, :] = temp_sol[21, :]
    gLNS_matrix[i, :]  = temp_sol[30, :]
    gLCa_matrix[i, :]  = temp_sol[31, :]
    
    gNtype_matrix[i, :] = temp_sol[34, :]
    gTtype_matrix[i, :] = temp_sol[35, :]
    Ca_ma_matrix[i, :] = moving_average(sol_rigorous(tt)[13, :], window_size_, padding_)
    
    # 1. Analyse AVANT BLOCAGE
    t_pre = win_pre[1]:0.1:win_pre[2]  
    sol_pre = sol_rigorous(t_pre)      
    f_pre = extract_frequency(sol_pre[1, :], t_pre)
    freqs_pre[i] = isnan(f_pre) ? 0.0 : f_pre 

    # 2. Analyse AIGU
    t_acute = win_acute[1]:0.1:win_acute[2]
    sol_acute = sol_rigorous(t_acute)
    f_acute = extract_frequency(sol_acute[1, :], t_acute)
    freqs_acute[i] = isnan(f_acute) ? 0.0 : f_acute

    # 3. Analyse COMPENSÉ 
    t_comp = win_comp[1]:0.1:win_comp[2]
    sol_comp = sol_rigorous(t_comp)
    f_comp = extract_frequency(sol_comp[1, :], t_comp)
    freqs_comp[i] = isnan(f_comp) ? 0.0 : f_comp
    
    
end

In [ ]:
# --- 1. PARAMÈTRES DE FILTRAGE ---
seuil_bas = 0.5  
seuil_haut = 15.0 
myTeal = RGBA(0/255, 128/255, 128/255, 1)

n_simulated = length(freqs_pre) 

is_valid_pm(f) = seuil_bas < f <= seuil_haut

# --- 2. CALCUL DES POURCENTAGES ---
pc_pre   = count(is_valid_pm, freqs_pre) / n_simulated * 100
pc_acute = count(is_valid_pm, freqs_acute) / n_simulated * 100
pc_comp  = count(is_valid_pm, freqs_comp) / n_simulated * 100

println("Résultats de la population ($n_simulated neurones) :")
println("- Avant blocage : $pc_pre % sont des pacemakers valides (0.5-15 Hz)")
println("- Juste après blocage : $pc_acute % sont des pacemakers valides")
println("- Après compensation : $pc_comp % sont des pacemakers valides")

# --- 3. LES GRAPHIQUES ---
h1 = histogram(freqs_pre, title="Pré-blocage ($pc_pre%)", color=myApple, label="", xlims=(0, 15))
h2 = histogram(freqs_acute, title="Aigu ($pc_acute%)", color=:red, label="", xlims=(0, 15))
h3 = histogram(freqs_comp, title="Compensé ($pc_comp%)", color=myTeal, label="", xlims=(0, 15))



plot(h1, h2, h3, layout=(1,3), size=(1100, 400), xlabel="Fréquence (Hz)", ylabel="Nombre de neurones")




In [ ]:
# --- 1. FILTRAGE DE FREQS_PRE ---
freqs_pre_filtered = freqs_pre[freqs_pre .<= 5.0]

# --- 2. CALCUL DES POURCENTAGES  ---
n_total = length(freqs_pre)
pc_pre   = count(f -> 0.5 < f <= 5.0, freqs_pre) / n_total * 100
pc_acute = count(is_valid_pm, freqs_acute) / n_total * 100
pc_comp  = count(is_valid_pm, freqs_comp) / n_total * 100

# --- 3. LES VIOLIN PLOTS ---
v1 = violin([freqs_pre_filtered], label="", color=myApple, grid=false,
            title="Pre-block  : $pc_pre%",
            xticks=([1,], [L"t_\mathrm{pre}",]), ylims=(0, 15))

v2 = violin([freqs_acute], label="", color=:red, grid=false,
            title="Acute (50sec) : $pc_acute%",
            xticks=([1,], [L"t_\mathrm{acute}",]), ylims=(0, 15))

v3 = violin([freqs_comp], label="", color=myTeal, grid=false,
            title="Compensated (300sec): $pc_comp%",
            xticks=([1,], [L"t_\mathrm{comp}",]), ylims=(0, 15))

plot(v1, v2, v3, layout=(1,3), size=(1100, 400), ylabel="Frequency (Hz)")

# **A-type Potassium Channel Blockade — Model Validation**

In [ ]:
# 3. Lancement de la population

Tfinal = 300000
tt = 0.0 : 0.2 : Tfinal
tspan  = (0.0, Tfinal)

window_size_s = 5.0       
padding_s = 0.2           
window_size_  = Int64(round(window_size_s * 1000 / 0.2)) 
padding_      = Int64(round(padding_s * 1000 / 0.2))
tt_moving_average_plot = range(window_size_s/2, Tfinal/1000 - window_size_s/2, length=length(moving_average(zeros(length(tt)), window_size_, padding_)))

tau_g = 0.1
t_Na = 0.6

ncells= 200
Tfinal = tspan[2]           

sample_indices = 2:5000:length(tt)
data_length = length(sample_indices)

gNa_matrix = zeros(ncells, data_length)
gCaL_matrix = zero(gNa_matrix)
gKd_matrix = zero(gNa_matrix)
gKA_matrix = zero(gNa_matrix)
gKERG_matrix = zero(gNa_matrix)
gKSK_matrix = zero(gNa_matrix)
gH_matrix = zero(gNa_matrix)
gPace_matrix = zero(gNa_matrix)
gLNS_matrix = zero(gNa_matrix)
gLCa_matrix = zero(gNa_matrix)
gNtype_matrix = zero(gNa_matrix)
gTtype_matrix = zero(gNa_matrix)

freqs_pre = zeros(ncells)
freqs_acute = zeros(ncells)
freqs_comp = zeros(ncells)

win_pre   = (5000, 25000)    
win_acute = (35000, 55000)   
win_comp  = (250000, 295000) 

Ca_ma_matrix = zeros(ncells, length(tt_moving_average_plot))

@showprogress "Computing Population..." for i = 1 : ncells
    gNa = g_all_init[i,1]
    gCaL = g_all_init[i,2]
    gKd = g_all_init[i,3]
    gKA = g_all_init[i,4]
    gKERG = g_all_init[i,5]
    gKSK = g_all_init[i,6]
    gH = g_all_init[i,7]
    gLNS = g_all_init[i,8]
    gLCa = g_all_init[i,9]
    gPace = g_all_init[i,10]
    gNtype = 0.2 
    gTtype = gTtype_final[i]
    
    Ca_tgt = Ca_tgt_moyen[i]
    
    tCaL   = t_Na * gNa / max(1e-6,gCaL)
    tKd    = t_Na * gNa / max(1e-6,gKd)
    tKA    = t_Na * gNa / max(1e-6,gKA)
    tKERG  = t_Na * gNa / max(1e-6,gKERG) 
    tKSK   = t_Na * gNa / max(1e-6,gKSK)
    tH     = t_Na * gNa / max(1e-6,gH)
    tPace  = t_Na * gNa / max(1e-6,gPace)
    tLNS   = t_Na * gNa / max(1e-6,gLNS)
    tLCa   = t_Na * gNa / max(1e-6,gLCa)
    tNCa   = t_Na * gNa / max(1e-6,gNtype)
    tTCA   = t_Na * gNa / max(1e-6,gTtype)
    
    Iapp_val(t) = 0.0
    
    g_main_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH,gPace]
    m_main_init = copy(g_main_init)

    g_leak_init = [gLNS,  gLCa]
    m_leak_init = copy(g_leak_init)

    g_calcique = [gNtype, gTtype]
    m_calcique = copy(g_calcique)
    
    g_max_main = g_main_init .* 10.0
    g_max_leak = g_leak_init .* 10.0
    g_max_cal  = g_calcique  .* 10.0
    
    g_max_all = (g_max_main, g_max_leak, g_max_cal)

    p_homeo = (Iapp_val, Ca_tgt, tau_g, t_Na, tCaL, tKd, tKA, tKERG, tKSK, 
           tH, tPace, tLNS, tLCa, tNCa, tTCA, g_max_all)

    V0 = -50.0
    Ca0 = Ca_tgt_moyen[i]
    
    u_biophys = [
        V0, m_inf_true(V0), h_inf(V0), hs_inf(V0), l_inf_true(V0), n_inf(V0), 
        p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
    ]

    uphys = [mN_inf(V0), hN_inf(V0), mT_inf(V0), hT_inf(V0)]

    u0_homeo = vcat(u_biophys, g_main_init, m_main_init, g_leak_init, m_leak_init, g_calcique, m_calcique, uphys)

    prob = ODEProblem(DA_homeo_ODE_Ablock, u0_homeo, tspan, p_homeo)
    
    sol_rigorous = solve(prob, Rodas5P(), reltol=1e-4, maxiters=1e7)
    
    if sol_rigorous.t[end] < Tfinal
        @warn "Crash persistant à t = $(sol_rigorous.t[end])"
    end
    
    t_echantillon = tt[sample_indices]
    temp_sol = sol_rigorous(t_echantillon)
    
    gNa_matrix[i, :]   = temp_sol[14, :]
    gCaL_matrix[i, :]  = temp_sol[15, :]
    gKd_matrix[i, :]   = temp_sol[16, :]
    gKA_regulee = temp_sol[17, :]
    gKA_matrix[i, :] = [ (t_echantillon[j] < 30000.0) ? gKA_regulee[j] : 1e-18 for j in 1:length(t_echantillon) ]
    gKERG_matrix[i, :] = temp_sol[18, :]
    gKSK_matrix[i, :]  = temp_sol[19, :]
    gH_matrix[i, :]    = temp_sol[20, :]
    gPace_matrix[i, :] = temp_sol[21, :]
    gLNS_matrix[i, :]  = temp_sol[30, :]
    gLCa_matrix[i, :]  = temp_sol[31, :]
    
    gNtype_matrix[i, :] = temp_sol[34, :]
    gTtype_matrix[i, :] = temp_sol[35, :]
    Ca_ma_matrix[i, :] = moving_average(sol_rigorous(tt)[13, :], window_size_, padding_)
    
    # 1. Analyse AVANT BLOCAGE
    t_pre = win_pre[1]:0.1:win_pre[2]  
    sol_pre = sol_rigorous(t_pre)      
    f_pre = extract_frequency(sol_pre[1, :], t_pre)
    freqs_pre[i] = isnan(f_pre) ? 0.0 : f_pre 

    # 2. Analyse AIGU 
    t_acute = win_acute[1]:0.1:win_acute[2]
    sol_acute = sol_rigorous(t_acute)
    f_acute = extract_frequency(sol_acute[1, :], t_acute)
    freqs_acute[i] = isnan(f_acute) ? 0.0 : f_acute

    # 3. Analyse COMPENSÉ 
    t_comp = win_comp[1]:0.1:win_comp[2]
    sol_comp = sol_rigorous(t_comp)
    f_comp = extract_frequency(sol_comp[1, :], t_comp)
    freqs_comp[i] = isnan(f_comp) ? 0.0 : f_comp
        
end

In [ ]:
# --- 1. PARAMÈTRES DE FILTRAGE ---
seuil_bas = 0.5  
seuil_haut = 15.0 
myTeal = RGBA(0/255, 128/255, 128/255, 1)

n_simulated = length(freqs_pre) 

is_valid_pm(f) = seuil_bas < f <= seuil_haut

# --- 2. CALCUL DES POURCENTAGES ---
pc_pre   = count(is_valid_pm, freqs_pre) / n_simulated * 100
pc_acute = count(is_valid_pm, freqs_acute) / n_simulated * 100
pc_comp  = count(is_valid_pm, freqs_comp) / n_simulated * 100

println("Résultats de la population ($n_simulated neurones) :")
println("- Avant blocage : $pc_pre % sont des pacemakers valides (0.5-15 Hz)")
println("- Juste après blocage : $pc_acute % sont des pacemakers valides")
println("- Après compensation : $pc_comp % sont des pacemakers valides")

# --- 3. LES GRAPHIQUES ---
h1 = histogram(freqs_pre, title="Pré-blocage ($pc_pre%)", color=myApple, label="", xlims=(0, 15))
h2 = histogram(freqs_acute, title="Aigu ($pc_acute%)", color=:red, label="", xlims=(0, 15))
h3 = histogram(freqs_comp, title="Compensé ($pc_comp%)", color=myTeal, label="", xlims=(0, 15))

plot(h1, h2, h3, layout=(1,3), size=(1100, 400), xlabel="Fréquence (Hz)", ylabel="Nombre de neurones")


In [ ]:
# --- 1. FILTRAGE DE FREQS_PRE ---
freqs_pre_filtered = freqs_pre[freqs_pre .<= 5.0]
is_valid_pm(f) = seuil_bas < f <= seuil_haut

# --- 2. CALCUL DES POURCENTAGES ---
n_total = length(freqs_pre)
pc_pre   = count(f -> 0.5 < f <= 5.0, freqs_pre) / n_total * 100
pc_acute = count(is_valid_pm, freqs_acute) / n_total * 100
pc_comp  = count(is_valid_pm, freqs_comp) / n_total * 100

# --- 3. LES VIOLIN PLOTS ---
v1 = violin([freqs_pre_filtered], label="", color=myApple, grid=false,
            title="Pre-block : $pc_pre%",
            xticks=([1,], [L"t_\mathrm{pre}",]), ylims=(0, 15))

v2 = violin([freqs_acute], label="", color=:red, grid=false,
            title="Acute (50sec) : $pc_acute%",
            xticks=([1,], [L"t_\mathrm{acute}",]), ylims=(0, 15))

v3 = violin([freqs_comp], label="", color=myTeal, grid=false,
            title="Compensated (300sec): $pc_comp%",
            xticks=([1,], [L"t_\mathrm{comp}",]), ylims=(0, 15))

plot(v1, v2, v3, layout=(1,3), size=(1100, 400), ylabel="Frequency (Hz)")

# **gPace Conductance Blockade (IXG current)**

In [ ]:
# 3. Lancement de la population 

Tfinal = 300000
tt = 0.0 : 0.2 : Tfinal
tspan  = (0.0, Tfinal)

window_size_s = 5.0       
padding_s = 0.2           
window_size_  = Int64(round(window_size_s * 1000 / 0.2)) 
padding_      = Int64(round(padding_s * 1000 / 0.2))
tt_moving_average_plot = range(window_size_s/2, Tfinal/1000 - window_size_s/2, length=length(moving_average(zeros(length(tt)), window_size_, padding_)))

tau_g = 0.1
t_Na = 0.6

ncells= 200
Tfinal = tspan[2]           

sample_indices = 2:5000:length(tt)
data_length = length(sample_indices)

gNa_matrix = zeros(ncells, data_length)
gCaL_matrix = zero(gNa_matrix)
gKd_matrix = zero(gNa_matrix)
gKA_matrix = zero(gNa_matrix)
gKERG_matrix = zero(gNa_matrix)
gKSK_matrix = zero(gNa_matrix)
gH_matrix = zero(gNa_matrix)
gPace_matrix = zero(gNa_matrix)
gLNS_matrix = zero(gNa_matrix)
gLCa_matrix = zero(gNa_matrix)
gNtype_matrix = zero(gNa_matrix)
gTtype_matrix = zero(gNa_matrix)

freqs_pre = zeros(ncells)
freqs_acute = zeros(ncells)
freqs_comp = zeros(ncells)

win_pre   = (5000, 25000)   
win_acute = (35000, 55000)   
win_comp  = (250000, 295000) 

Ca_ma_matrix = zeros(ncells, length(tt_moving_average_plot))

@showprogress "Computing Population..." for i = 1 : ncells
    gNa = g_all_init[i,1]
    gCaL = g_all_init[i,2]
    gKd = g_all_init[i,3]
    gKA = g_all_init[i,4]
    gKERG = g_all_init[i,5]
    gKSK = g_all_init[i,6]
    gH = g_all_init[i,7]
    gLNS = g_all_init[i,8]
    gLCa = g_all_init[i,9]
    gPace = g_all_init[i,10]
    gNtype = 0.2 *(0.9 + 0.2 * rand())
    gTtype = gTtype_final[i]
    
    Ca_tgt = Ca_tgt_moyen[i]
    
    tCaL   = t_Na * gNa / max(1e-6,gCaL)
    tKd    = t_Na * gNa / max(1e-6,gKd)
    tKA    = t_Na * gNa / max(1e-6,gKA)
    tKERG  = t_Na * gNa / max(1e-6,gKERG) 
    tKSK   = t_Na * gNa / max(1e-6,gKSK)
    tH     = t_Na * gNa / max(1e-6,gH)
    tPace  = t_Na * gNa / max(1e-6,gPace)
    tLNS   = t_Na * gNa / max(1e-6,gLNS)
    tLCa   = t_Na * gNa / max(1e-6,gLCa)
    tNCa   = t_Na * gNa / max(1e-6,gNtype)
    tTCA   = t_Na * gNa / max(1e-6,gTtype)

    Iapp_val(t) = 0.0
    
    g_main_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH,gPace]
    m_main_init = copy(g_main_init)

    g_leak_init = [gLNS,  gLCa]
    m_leak_init = copy(g_leak_init)

    g_calcique = [gNtype, gTtype]
    m_calcique = copy(g_calcique)
    
    g_max_main = g_main_init .* 10.0
    g_max_leak = g_leak_init .* 10.0
    g_max_cal  = g_calcique  .* 10.0

    g_max_all = (g_max_main, g_max_leak, g_max_cal)

    p_homeo = (Iapp_val, Ca_tgt, tau_g, t_Na, tCaL, tKd, tKA, tKERG, tKSK, 
           tH, tPace, tLNS, tLCa, tNCa, tTCA, g_max_all)

    V0 = -50.0
    Ca0 = Ca_tgt_moyen[i]
    
    u_biophys = [
        V0, m_inf_true(V0), h_inf(V0), hs_inf(V0), l_inf_true(V0), n_inf(V0), 
        p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
    ]

    uphys = [mN_inf(V0), hN_inf(V0), mT_inf(V0), hT_inf(V0)]
 
    u0_homeo = vcat(u_biophys, g_main_init, m_main_init, g_leak_init, m_leak_init, g_calcique, m_calcique, uphys)

    prob = ODEProblem(DA_homeo_gPace, u0_homeo, tspan, p_homeo)
    sol_rigorous = solve(prob, Rodas5P(), reltol=1e-4, maxiters=1e7)
    
    #sol_rigorous = solve(prob; maxiters=1e7)
    
    if sol_rigorous.t[end] < Tfinal
        @warn "Crash persistant à t = $(sol_rigorous.t[end])"
    end
    
    t_echantillon = tt[sample_indices]
    temp_sol = sol_rigorous(t_echantillon)
    
    gNa_matrix[i, :]   = temp_sol[14, :]
    gCaL_matrix[i, :]  = temp_sol[15, :]
    gKd_matrix[i, :]   = temp_sol[16, :]
    gKA_matrix[i, :]   = temp_sol[17, :]
    gKERG_matrix[i, :] = temp_sol[18, :]
    gKSK_matrix[i, :]  = temp_sol[19, :]
    gH_matrix[i, :]    = temp_sol[20, :]
    gPace_regulee = temp_sol[21, :]
    gPace_matrix[i, :] = [ (t_echantillon[j] < 30000.0) ? gPace_regulee[j] : 1e-18 for j in 1:length(t_echantillon) ]
    
    gLNS_matrix[i, :]  = temp_sol[30, :]
    gLCa_matrix[i, :]  = temp_sol[31, :]
    
    gNtype_matrix[i, :] = temp_sol[34, :]
    gTtype_matrix[i, :] = temp_sol[35, :]
    Ca_ma_matrix[i, :] = moving_average(sol_rigorous(tt)[13, :], window_size_, padding_)
  
    # 1. Analyse AVANT BLOCAGE
    t_pre = win_pre[1]:0.1:win_pre[2]  
    sol_pre = sol_rigorous(t_pre)      
    f_pre = extract_frequency(sol_pre[1, :], t_pre)
    freqs_pre[i] = isnan(f_pre) ? 0.0 : f_pre 

    # 2. Analyse AIGU 
    t_acute = win_acute[1]:0.1:win_acute[2]
    sol_acute = sol_rigorous(t_acute)
    f_acute = extract_frequency(sol_acute[1, :], t_acute)
    freqs_acute[i] = isnan(f_acute) ? 0.0 : f_acute

    # 3. Analyse COMPENSÉ 
    t_comp = win_comp[1]:0.1:win_comp[2]
    sol_comp = sol_rigorous(t_comp)
    f_comp = extract_frequency(sol_comp[1, :], t_comp)
    freqs_comp[i] = isnan(f_comp) ? 0.0 : f_comp
    
    #Graphique activité électrique
    #t_zoom1 = 5000 : 0.1 : 7000
    #res_zoom1 = sol_rigorous(t_zoom1)
    #p_v1 = plot(t_zoom1 ./ 1e3, res_zoom1[1, :], linewidth=1.5, color=myDarkBlue, 
    #legend=false, ylims=(-90, 30), xlims=(5, 7), 
    #title="Electrical Trace 5-7 sec ",
    #xticks=([5, 6, 7], ["5", "6", "7"]))
    #ylabel!("V (mV)")

    #t_zoom2 = 51000 : 0.1 : 53000 
    #res_zoom2 = sol_rigorous(t_zoom2) # Le solveur recalcule les points exacts

    # 2. On plot avec ce nouveau vecteur
    #p_v_201 = plot(t_zoom2 ./ 1e3, res_zoom2[1, :], linewidth=1.5, color=myDarkBlue, 
    #legend=false, ylims=(-90, 30), xlims=(51, 53), 
    #title="Electrical Trace 51-53 sec",
    #xticks=([51, 52, 53], ["51", "52", "53"]))
    #ylabel!("V (mV)")
    
    # 1. On crée un échantillonnage TRÈS FIN juste pour le zoom (0.1 ms)
    #t_zoom = 291000 : 0.1 :293000 
    #res_zoom = sol_rigorous(t_zoom) # Le solveur recalcule les points exacts

    # 2. On plot avec ce nouveau vecteur
    #p_v_2014 = plot(t_zoom ./ 1e3, res_zoom[1, :], linewidth=1.5, color=myDarkBlue, 
    #legend=false, ylims=(-90, 30), xlims=(291, 293), 
    #title="Electrical Trace 292-293 sec",
    #xticks=([291, 292, 293], ["291", "292", "293"]))
    #ylabel!("V (mV)")
    #xlabel!("Time (s)")
    
    #l = @layout [a{0.33h}; b{0.33h}; c{0.34h}]
    #pu = plot(p_v1, p_v_201, p_v_2014, layout=l, size=(1000, 1000), plot_title = "Neuron $i",
          #plot_titlefontsize = 20,
          #plot_titlevspan = 0.08,
        #margin=10Plots.px)
    #display(pu)
        
end

In [ ]:
# --- 1. FILTRAGE DE FREQS_PRE ---
# On ne garde que les neurones qui spikent à moins de 5 Hz en condition initiale
freqs_pre_filtered = freqs_pre[freqs_pre .<= 5.0]
is_valid_pm(f) = seuil_bas < f <= seuil_haut

# --- 2. CALCUL DES POURCENTAGES (basé sur le total initial) ---
n_total = length(freqs_pre)
pc_pre   = count(f -> 0.5 < f <= 5.0, freqs_pre) / n_total * 100
pc_acute = count(is_valid_pm, freqs_acute) / n_total * 100
pc_comp  = count(is_valid_pm, freqs_comp) / n_total * 100

# --- 3. LES VIOLIN PLOTS ---
# Note : on utilise freqs_pre_filtered uniquement pour le premier plot
v1 = violin([freqs_pre_filtered], label="", color=myApple, grid=false,
            title="Pre-blockade  : $pc_pre%",
            xticks=([1,], [L"t_\mathrm{pre}",]), ylims=(0, 15))

v2 = violin([freqs_acute], label="", color=:red, grid=false,
            title="Acute (50sec) : 7%",
            xticks=([1,], [L"t_\mathrm{acute}",]), ylims=(0, 15))

v3 = violin([freqs_comp], label="", color=myTeal, grid=false,
            title="Compensated (300sec): 3.5%",
            xticks=([1,], [L"t_\mathrm{comp}",]), ylims=(0, 15))

plot(v1, v2, v3, layout=(1,3), size=(1100, 400), ylabel="Frequency (Hz)")

# **gPace Blockade + Stochastic Synaptic Noise**

In [ ]:
struct NoisyFunction
    amplitude::Float64 
end

function (nf::NoisyFunction)(x)
    noise = nf.amplitude * randn()  
    return noise  
end

In [ ]:
# 3. Lancement de la population

Tfinal = 300000
tt = 0.0 : 0.2 : Tfinal
tspan  = (0.0, Tfinal)

window_size_s = 5.0       
padding_s = 0.2           
window_size_  = Int64(round(window_size_s * 1000 / 0.2)) 
padding_      = Int64(round(padding_s * 1000 / 0.2))
tt_moving_average_plot = range(window_size_s/2, Tfinal/1000 - window_size_s/2, length=length(moving_average(zeros(length(tt)), window_size_, padding_)))

tau_g = 0.1
t_Na = 0.6

ncells= 151
Tfinal = tspan[2]           

sample_indices = 2:5000:length(tt)
data_length = length(sample_indices)

gNa_matrix = zeros(ncells, data_length)
gCaL_matrix = zero(gNa_matrix)
gKd_matrix = zero(gNa_matrix)
gKA_matrix = zero(gNa_matrix)
gKERG_matrix = zero(gNa_matrix)
gKSK_matrix = zero(gNa_matrix)
gH_matrix = zero(gNa_matrix)
gPace_matrix = zero(gNa_matrix)
gLNS_matrix = zero(gNa_matrix)
gLCa_matrix = zero(gNa_matrix)
gNtype_matrix = zero(gNa_matrix)
gTtype_matrix = zero(gNa_matrix)

freqs_pre = zeros(ncells)
freqs_acute = zeros(ncells)
freqs_comp = zeros(ncells)

win_pre   = (5000, 25000)    
win_acute = (35000, 55000)   
win_comp  = (250000, 295000) 

Ca_ma_matrix = zeros(ncells, length(tt_moving_average_plot))

@showprogress "Computing Population..." for i = 150 : ncells
    gNa = g_all_init[i,1]
    gCaL = g_all_init[i,2]
    gKd = g_all_init[i,3]
    gKA = g_all_init[i,4]
    gKERG = g_all_init[i,5]
    gKSK = g_all_init[i,6]
    gH = g_all_init[i,7]
    gLNS = g_all_init[i,8]
    gLCa = g_all_init[i,9]
    gPace = g_all_init[i,10]
    gNtype = 0.2 *(0.9 + 0.2 * rand())
    gTtype = gTtype_final[i]
    
    Ca_tgt = Ca_tgt_moyen[i]
    
    tCaL   = t_Na * gNa / max(1e-6,gCaL)
    tKd    = t_Na * gNa / max(1e-6,gKd)
    tKA    = t_Na * gNa / max(1e-6,gKA)
    tKERG  = t_Na * gNa / max(1e-6,gKERG) 
    tKSK   = t_Na * gNa / max(1e-6,gKSK)
    tH     = t_Na * gNa / max(1e-6,gH)
    tPace  = t_Na * gNa / max(1e-6,gPace)
    tLNS   = t_Na * gNa / max(1e-6,gLNS)
    tLCa   = t_Na * gNa / max(1e-6,gLCa)
    tNCa   = t_Na * gNa / max(1e-6,gNtype)
    tTCA   = t_Na * gNa / max(1e-6,gTtype)
    
    amplitude_bruit =15
    Iapp_noisy = NoisyFunction(amplitude_bruit)
    
    g_main_init = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH,gPace]
    m_main_init = copy(g_main_init)

    g_leak_init = [gLNS,  gLCa]
    m_leak_init = copy(g_leak_init)

    g_calcique = [gNtype, gTtype]
    m_calcique = copy(g_calcique)
    
    g_max_main = g_main_init .* 10.0
    g_max_leak = g_leak_init .* 10.0
    g_max_cal  = g_calcique  .* 10.0

    g_max_all = (g_max_main, g_max_leak, g_max_cal)

    p_homeo = (Iapp_noisy, Ca_tgt, tau_g, t_Na, tCaL, tKd, tKA, tKERG, tKSK, 
           tH, tPace, tLNS, tLCa, tNCa, tTCA, g_max_all)

    V0 = -50.0
    Ca0 = Ca_tgt_moyen[i]
   
    u_biophys = [
        V0, m_inf_true(V0), h_inf(V0), hs_inf(V0), l_inf_true(V0), n_inf(V0), 
        p_inf(V0), q1_inf(V0), q2_inf(V0), 0.0, 0.0, mH_inf(V0), Ca0
    ]

    uphys = [mN_inf(V0), hN_inf(V0), mT_inf(V0), hT_inf(V0)]

    u0_homeo = vcat(u_biophys, g_main_init, m_main_init, g_leak_init, m_leak_init, g_calcique, m_calcique, uphys)

    prob = ODEProblem(DA_homeo_gPace, u0_homeo, tspan, p_homeo)
    sol_rigorous = solve(prob, Rodas5P(), reltol=1e-4, maxiters=1e7)
  
    #sol_rigorous = solve(prob; maxiters=1e7)
    
    if sol_rigorous.t[end] < Tfinal
        @warn "Crash persistant à t = $(sol_rigorous.t[end])"
    end
    
    t_echantillon = tt[sample_indices]
    temp_sol = sol_rigorous(t_echantillon)
    
    gNa_matrix[i, :]   = temp_sol[14, :]
    gCaL_matrix[i, :]  = temp_sol[15, :]
    gKd_matrix[i, :]   = temp_sol[16, :]
    gKA_matrix[i, :]   = temp_sol[17, :]
    gKERG_matrix[i, :] = temp_sol[18, :]
    gKSK_matrix[i, :]  = temp_sol[19, :]
    gH_matrix[i, :]    = temp_sol[20, :]
    
    gPace_regulee = temp_sol[21, :]
    gPace_matrix[i, :] = [ (t_echantillon[j] < 30000.0) ? gPace_regulee[j] : 1e-18 for j in 1:length(t_echantillon) ]
    
    gLNS_matrix[i, :]  = temp_sol[30, :]
    gLCa_matrix[i, :]  = temp_sol[31, :]
    
    gNtype_matrix[i, :] = temp_sol[34, :]
    gTtype_matrix[i, :] = temp_sol[35, :]
    Ca_ma_matrix[i, :] = moving_average(sol_rigorous(tt)[13, :], window_size_, padding_)
  
    # 1. Analyse AVANT BLOCAGE
    t_pre = win_pre[1]:0.1:win_pre[2]  
    sol_pre = sol_rigorous(t_pre)      
    f_pre = extract_frequency(sol_pre[1, :], t_pre)
    freqs_pre[i] = isnan(f_pre) ? 0.0 : f_pre 

    # 2. Analyse AIGU 
    t_acute = win_acute[1]:0.1:win_acute[2]
    sol_acute = sol_rigorous(t_acute)
    f_acute = extract_frequency(sol_acute[1, :], t_acute)
    freqs_acute[i] = isnan(f_acute) ? 0.0 : f_acute

    # 3. Analyse COMPENSÉ 
    t_comp = win_comp[1]:0.1:win_comp[2]
    sol_comp = sol_rigorous(t_comp)
    f_comp = extract_frequency(sol_comp[1, :], t_comp)
    freqs_comp[i] = isnan(f_comp) ? 0.0 : f_comp
      
        
end

In [ ]:
 #plot calcium neurone individuel 
# --- CHOIX DU NEURONE ---
idx = 151 

# --- PLOT CALCIUM ---
p_ca = plot(size=(800, 400), title="Calcium Convergence - Neuron $idx")

# Courbe du calcium (Moyenne mobile)
plot!(tt_moving_average_plot, Ca_ma_matrix[idx, :], 
      linewidth=2, color=:black, label="[Ca] mean")

# Ligne cible
hline!([Ca_tgt_moyen[idx]], color=:firebrick1, linestyle=:dash, 
       linewidth=2, label="Target")

ylabel!("[Ca]")
xlabel!("t(s)")
yaxis!(:log) 

display(p_ca)

In [ ]:
# --- 1. FILTRAGE DE FREQS_PRE ---
# On ne garde que les neurones qui spikent à moins de 5 Hz en condition initiale
freqs_pre_filtered = freqs_pre[freqs_pre .<= 5.0]
is_valid_pm(f) = seuil_bas < f <= seuil_haut

# --- 2. CALCUL DES POURCENTAGES (basé sur le total initial) ---
n_total = length(freqs_pre)
pc_pre   = count(f -> 0.5 < f <= 5.0, freqs_pre) / n_total * 100
pc_acute = count(is_valid_pm, freqs_acute) / n_total * 100
pc_comp  = count(is_valid_pm, freqs_comp) / n_total * 100

# --- 3. LES VIOLIN PLOTS ---
# Note : on utilise freqs_pre_filtered uniquement pour le premier plot
v1 = violin([freqs_pre_filtered], label="", color=myApple, grid=false,
            title="Pre-blockade  : $pc_pre%",
            xticks=([1,], [L"t_\mathrm{pre}",]), ylims=(0, 15))

v2 = violin([freqs_acute], label="", color=:red, grid=false,
            title="Acute (50sec) : $pc_acute%",
            xticks=([1,], [L"t_\mathrm{acute}",]), ylims=(0, 15))

v3 = violin([freqs_comp], label="", color=myTeal, grid=false,
            title="Compensated (300sec): $pc_comp%",
            xticks=([1,], [L"t_\mathrm{comp}",]), ylims=(0, 15))

plot(v1, v2, v3, layout=(1,3), size=(1100, 400), ylabel="Fréquency (Hz)")